# 01 — Data Exploration

**Goals**
- Load and inspect `Test.csv`
- Check even sampling and sampling frequency (Q1)
- Plot raw Acceleration and Load time series

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from well_analysis.data import load_test_data
from well_analysis.signal import check_even_sampling, compute_intervals
from well_analysis.viz import plot_time_series

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

## 1. Load data

In [ ]:
df = load_test_data()
print(f"Rows: {len(df):,}")
print(f"Time span: {df['Timestamp'].iloc[0]}  →  {df['Timestamp'].iloc[-1]}")
df.head()

In [ ]:
df.describe()

## 2. Even sampling? (Q1)

Efficient method: compute successive timestamp differences, take the median as the nominal interval, then check that the max deviation is negligible.

In [ ]:
is_even, fs = check_even_sampling(df['Timestamp'])
intervals = compute_intervals(df['Timestamp'])

print(f"Evenly sampled : {is_even}")
print(f"Sampling freq  : {fs:.4f} Hz  (Δt = {1/fs*1000:.2f} ms)")
print(f"Interval stats : min={intervals.min():.4f}s  max={intervals.max():.4f}s  std={intervals.std():.2e}s")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(intervals, lw=0.5)
ax.set_xlabel('Sample index')
ax.set_ylabel('Δt (s)')
ax.set_title('Successive timestamp intervals')
plt.tight_layout()
plt.show()

## 3. Raw signal overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
plot_time_series(df['Timestamp'], df['Acceleration'], ylabel='Acceleration (m/s²)',
                 title='Acceleration — full dataset', ax=axes[0])
plot_time_series(df['Timestamp'], df['Load'], ylabel='Load (N)',
                 title='Load — full dataset', ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Zoom on a short running window (~5 min)

In [ ]:
# TODO: pick a time window once we know the dataset dates
window_start = df['Timestamp'].iloc[0]
window_end = window_start + pd.Timedelta(minutes=5)
sub = df[(df['Timestamp'] >= window_start) & (df['Timestamp'] < window_end)]

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
plot_time_series(sub['Timestamp'], sub['Acceleration'], ylabel='m/s²', ax=axes[0])
plot_time_series(sub['Timestamp'], sub['Load'], ylabel='N', ax=axes[1])
plt.tight_layout()
plt.show()